# 1. Environment Setup



In [1]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which

In [2]:
# Clone CCLUE datasets
!git clone https://github.com/Ethan-yt/CCLUE.git

Cloning into 'CCLUE'...
remote: Enumerating objects: 708, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 708 (delta 8), reused 5 (delta 5), pack-reused 694 (from 1)
Receiving objects: 100% (708/708), 59.18 MiB | 10.02 MiB/s, done.
Resolving deltas: 100% (128/128), done.


In [3]:
# Verify dataset structure
!ls CCLUE/data/fspc

dev.tsv  fspc.py  raw.json  test.tsv  train.tsv


# 2. Import libraries

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import ast
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer, DataCollatorWithPadding, TrainerCallback
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
import os, gc
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")



True
NVIDIA L4


In [5]:
# Configuration
MODEL_NAME = "BAAI/bge-base-zh-v1.5"
MAX_LEN = 128
NUM_LABELS = 10  # Change to 10 for CLS dataset

LINGUISTIC_FEATURE = [
    'mid_initial_vector', 'mid_final_vector', 'mid_tone_vector',
    'old_initial_vector', 'old_nucleus_vector', 'old_coda_vector',
    'man_initial_vector', 'man_final_vector', 'man_tone_vector',
    'can_onset_vector', 'can_nucleus_vector', 'can_coda_vector', 'can_tone_vector',
    'seal_radical_vector', 'trad_radical_vector', 'simp_radical_vector', 'oracle_radical'
]
MANDARIN = [6,7,8,14,15]
CLASSIC = [0,1,2,3,4,5,13,14,16]
MODERN = [6,7,8,9,10,11,12,14,15]

import os
os.environ["WANDB_DISABLED"] = "true"

# 3. Load Linguistic Features

In [11]:
def make_linguistic_dict(features=None):
    columns = []
    if features is None:
        columns = LINGUISTIC_FEATURE
    else:
        columns = [LINGUISTIC_FEATURE[f] for f in features]
    dim = 0
    df = pd.read_csv("Chinese_linguistic_data.csv")
    for col in columns:
        df[col] = df[col].apply(ast.literal_eval)
        dim += len(df[col][0])
    df['full_vector'] = df[columns].apply(lambda row: list(np.concatenate(row.values)), axis=1)
    char_to_vector = dict(zip(df['char'], df['full_vector']))
    del df
    gc.collect()
    return char_to_vector, dim

# 4. Model Architecture

In [12]:
class LIBGEWrapper(nn.Module):
    def __init__(self, model_name, linguistic_map=None, feature_dims=2007):
        super().__init__()
        self.model = AutoModel.from_pretrained(model_name)
        for param in self.model.parameters():
            param.requires_grad = False
        self.feature_dims = feature_dims
        self.fusion = nn.Sequential(
            nn.Linear(feature_dims, self.model.config.hidden_size),
            nn.ReLU(),
            nn.Dropout(0.1)
        )
        self.linguistic_map = linguistic_map

    def char_to_vector(self, char):
        return torch.tensor(self.linguistic_map.get(char, [0]*self.feature_dims), dtype=torch.float32)

    def forward(self, input_ids, attention_mask, raw_text):
        output = self.model(input_ids=input_ids, attention_mask=attention_mask)
        batch_vectors = []
        for sentence in raw_text:
            char_vecs = [self.char_to_vector(c).to(input_ids.device) for c in sentence][:output.last_hidden_state.size(1)]
            padding = [torch.zeros(self.feature_dims, device=input_ids.device)] * (output.last_hidden_state.size(1) - len(char_vecs))
            batch_vectors.append(torch.stack(char_vecs + padding))
        raw_text_vectors = torch.stack(batch_vectors)
        fused = self.fusion(raw_text_vectors)
        return (output.last_hidden_state + fused).mean(dim=1)


In [13]:
class TextClassifier(nn.Module):
    def __init__(self, encoder, num_labels):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(encoder.model.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, raw_text, labels=None):
        embeddings = self.encoder(input_ids, attention_mask, raw_text)
        logits = self.classifier(embeddings)
        labels = labels.long() if labels is not None else None
        loss = F.cross_entropy(logits, labels) if labels is not None else None
        return {'loss': loss, 'logits': logits} if loss else {'logits': logits}

# 5. Data Processing

In [33]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
ling_map,dim = make_linguistic_dict(CLASSIC)

In [8]:
# Initialize components
from sklearn.preprocessing import LabelEncoder

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt"
    )

def process_dataset(dataset):
    return dataset.map(lambda x: {
        "input_ids": tokenizer(
            x["text"],
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )["input_ids"][0],
        "attention_mask": tokenizer(
            x["text"],
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )["attention_mask"][0],
        "raw_text": list(x["text"]),
        "label": int(x["label"])  # <-- Shift to 0-indexed
    })
# Load dataset

import pandas as pd
from datasets import Dataset, DatasetDict

# Load into DataFrames
train_df = pd.read_csv("CCLUE/data/text_classification/train.tsv", sep="\t", names=["label", "text"], skiprows=1)
val_df   = pd.read_csv("CCLUE/data/text_classification/dev.tsv", sep="\t", names=["label", "text"], skiprows=1)
test_df  = pd.read_csv("CCLUE/data/text_classification/test.tsv", sep="\t", names=["label", "text"], skiprows=1)

label_encoder = LabelEncoder()
label_encoder.fit(train_df['label'])

train_df['label'] = label_encoder.transform(train_df['label'])
val_df['label'] = label_encoder.transform(val_df['label'])
test_df['label'] = label_encoder.transform(test_df['label'])
# Convert to HuggingFace Datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)
test_dataset  = Dataset.from_pandas(test_df)

# Bundle into DatasetDict
dataset = DatasetDict({
    "train": train_dataset,
    "val": val_dataset,
    "test": test_dataset
})

processed_ds = process_dataset(dataset)

Map:   0%|          | 0/160000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [14]:
class CustomCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.data_collator = DataCollatorWithPadding(tokenizer, return_tensors="pt")

    def __call__(self, features):
        raw_texts = [f.pop("raw_text") for f in features]  # keep raw_text separately
        batch = self.data_collator(features)
        batch["raw_text"] = raw_texts  # manually add it back
        return batch

collate_fn = CustomCollator(tokenizer)

class PrintLossCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            print(f"Epoch {int(state.epoch)} — Loss: {logs['loss']:.4f}")

# 6. Training Setup

In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }


In [15]:
# Initialize model

base_encoder = LIBGEWrapper(MODEL_NAME, linguistic_map=ling_map, feature_dims=dim)
model = TextClassifier(base_encoder, NUM_LABELS)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=0.001,
    per_device_train_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

from transformers import EarlyStoppingCallback
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_ds["train"],
    eval_dataset=processed_ds["val"],
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    callbacks=[PrintLossCallback(), EarlyStoppingCallback(early_stopping_patience=3)]
)


NameError: name 'ling_map' is not defined

# 7. Training & Evaluation

In [35]:
trainer.train()

# Final evaluation
test_results = trainer.evaluate(processed_ds["test"])
print("Test Results:", test_results)

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.903000,0.887613,0.711600,0.707931


Test Results: {'eval_loss': 0.8838710188865662, 'eval_accuracy': 0.7159, 'eval_f1_macro': 0.7123706648982878, 'eval_runtime': 447.0529, 'eval_samples_per_second': 44.737, 'eval_steps_per_second': 5.592, 'epoch': 1.0}


In [18]:
# evaluate base model
from sentence_transformers import SentenceTransformer
import torch.nn as nn
import torch.nn.functional as F

class BaseTextClassifier(nn.Module):
    def __init__(self, encoder, num_labels):
        super().__init__()
        self.encoder = encoder
        for param in self.encoder.parameters():
            param.requires_grad = False

        # Run a dummy input to get embedding size
        dummy_emb = self.encoder.encode("test", convert_to_tensor=True)
        hidden_size = dummy_emb.shape[-1]

        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        # SentenceTransformer doesn't use token-level input directly
        texts = [self.encoder.tokenizer.decode(ids[:mask.sum()].tolist()) for ids, mask in zip(input_ids, attention_mask)]
        embeddings = self.encoder.encode(texts, convert_to_tensor=True)

        logits = self.classifier(embeddings)
        labels = labels.long() if labels is not None else None
        loss = F.cross_entropy(logits, labels) if labels is not None else None
        return {'loss': loss, 'logits': logits} if loss is not None else {'logits': logits}

base_encoder = SentenceTransformer(MODEL_NAME)
model = BaseTextClassifier(base_encoder, NUM_LABELS)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer)

def process_dataset(dataset):
    return dataset.map(lambda x: {
        "input_ids": tokenizer(
            x["text"],
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )["input_ids"][0],
        "attention_mask": tokenizer(
            x["text"],
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )["attention_mask"][0],
        "label": int(x["label"])  # <-- Shift to 0-indexed
    })

processed_ds = process_dataset(dataset)
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=0.001,
    per_device_train_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_ds["train"],
    eval_dataset=processed_ds["val"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[PrintLossCallback()]
)
trainer.train()
test_results = trainer.evaluate(processed_ds["test"])
print("Test Results:", test_results)

Map:   0%|          | 0/160000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.028700,1.026373,0.667950,0.661961
2,0.988600,0.988566,0.680000,0.673482


KeyboardInterrupt: 